# Fine-tune Cross-Encoder v0.6 - Learning Rate Grid Search

Phase 2.1 of experimentation roadmap: Test 4 learning rates to find optimal LR for CV-JD matching.

| | |
|---|---|
| **Dataset** | v0.5 — 9,350 train / 2,000 validation / 2,000 test (13,350 pairs, balanced 20% per class) |
| **Base model** | `cross-encoder/ms-marco-MiniLM-L-12-v2` |
| **Loss** | `MSELoss` |
| **Evaluator** | Spearman correlation |
| **Epochs** | **15** (fixed from Phase 1) |
| **Batch size** | 16 (fixed baseline) |
| **Learning rates** | 1e-5, 2e-5, 3e-5, 5e-5 |
| **Branch** | `experiment/cross-encoder-v0.6` |

**Goal**: Identify optimal LR to maximize test LabelAcc (current baseline: 61.60% at LR=2e-5).

**Expected**: +1-2pp improvement → 62-63% target.

In [1]:
import os
if not os.path.exists('/content/Ai-Recruiter-Mini-Ai-Service'):
    !git clone https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service /content/Ai-Recruiter-Mini-Ai-Service

%cd /content/Ai-Recruiter-Mini-Ai-Service
!git checkout experiment/cross-encoder-v0.6
!git pull origin experiment/cross-encoder-v0.6

Cloning into '/content/Ai-Recruiter-Mini-Ai-Service'...
remote: Enumerating objects: 7364, done.
remote: Counting objects: 100% (356/356), done.
remote: Compressing objects: 100% (186/186), done.
remote: Total 7364 (delta 207), reused 191 (delta 169), pack-reused 7008 (from 2)
Receiving objects: 100% (7364/7364), 32.21 MiB | 7.37 MiB/s, done.
Resolving deltas: 100% (4372/4372), done.
/content/Ai-Recruiter-Mini-Ai-Service
Branch 'experiment/cross-encoder-v0.6' set up to track remote branch 'experiment/cross-encoder-v0.6' from 'origin'.
Switched to a new branch 'experiment/cross-encoder-v0.6'
From https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service
 * branch            experiment/cross-encoder-v0.6 -> FETCH_HEAD
Already up to date.


In [2]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 517.7/517.7 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.0/472.0 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.2/375.2 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA available: True
Device: Tesla T4


## Helper Functions

In [4]:
import json
import torch
import torch.nn.functional as F
import numpy as np
from scipy.stats import spearmanr
from sentence_transformers import CrossEncoder, InputExample
from typing import Any

# Evaluator: CECorrelationEvaluator (Spearman)
class CECorrelationEvaluator:
    """Evaluate cross-encoder using Spearman correlation."""
    def __init__(self, sentence_pairs: list, labels_0_1: list[float], name: str = ""):
        self.sentence_pairs = sentence_pairs
        self.labels_0_1 = labels_0_1
        self.name = name

    @classmethod
    def from_input_examples(cls, examples, name: str = "") -> "CECorrelationEvaluator":
        return cls(
            sentence_pairs=[ex.texts for ex in examples],
            labels_0_1=[ex.label for ex in examples],
            name=name,
        )

    def __call__(self, model, output_path=None, epoch: int = -1, steps: int = -1) -> float:
        preds = model.predict(self.sentence_pairs, batch_size=32, show_progress_bar=False)
        corr, _ = spearmanr(preds, self.labels_0_1)
        return float(corr) if not np.isnan(corr) else 0.0

def load_jsonl(path: str) -> list[dict[str, Any]]:
    """Load JSONL file."""
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

def load_dataset(data_dir: str) -> tuple[list[InputExample], list[InputExample], list[InputExample]]:
    """Load train/val/test InputExample lists."""
    train_data = load_jsonl(f"{data_dir}/cross_encoder_train.jsonl")
    val_data = load_jsonl(f"{data_dir}/cross_encoder_validation.jsonl")
    test_data = load_jsonl(f"{data_dir}/cross_encoder_test.jsonl")

    def to_input_examples(records):
        return [
            InputExample(texts=[rec['cv_text'], rec['jd_text']], label=rec['label'])
            for rec in records
        ]

    return to_input_examples(train_data), to_input_examples(val_data), to_input_examples(test_data)

def compute_metrics(model: CrossEncoder, examples: list[InputExample], batch_size: int = 32) -> dict:
    """Compute MAE, RMSE, LabelAcc on examples."""
    preds = model.predict([ex.texts for ex in examples], batch_size=batch_size, show_progress_bar=False)
    preds_100 = np.asarray(preds) * 100
    labels_100 = np.array([ex.label * 100 for ex in examples])

    mae = np.mean(np.abs(preds_100 - labels_100))
    rmse = np.sqrt(np.mean((preds_100 - labels_100) ** 2))
    label_acc = np.mean(np.abs(preds_100 - labels_100) <= 10)

    return {'MAE': mae, 'RMSE': rmse, 'LabelAcc': label_acc}

print("✅ Helper functions loaded.")

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


✅ Helper functions loaded.


## Load Dataset

In [5]:
train_examples, val_examples, test_examples = load_dataset('datasets/versions/v0.5/cross_encoder')

print(f"Train: {len(train_examples)} pairs")
print(f"Val:   {len(val_examples)} pairs")
print(f"Test:  {len(test_examples)} pairs")
print(f"Total: {len(train_examples) + len(val_examples) + len(test_examples)} pairs")

Train: 9350 pairs
Val:   2000 pairs
Test:  2000 pairs
Total: 13350 pairs


## Learning Rate Grid Search

In [6]:
import torch
from torch.utils.data import DataLoader
import os

# Configuration
learning_rates = [1e-5, 2e-5, 3e-5, 5e-5]
base_model = 'cross-encoder/ms-marco-MiniLM-L-12-v2'
epochs = 15
batch_size = 16

# Store all results
all_results = []

# Calculate warmup steps once (fixed for all runs)
total_steps = (len(train_examples) // batch_size + 1) * epochs
warmup_steps = int(total_steps * 0.1)

print(f"📊 Grid Search Configuration:")
print(f"  Learning rates: {learning_rates}")
print(f"  Epochs: {epochs}")
print(f"  Batch size: {batch_size}")
print(f"  Warmup steps: {warmup_steps}")
print(f"  Total steps per run: {total_steps}")
print()

for lr in learning_rates:
    print(f"\n{'='*60}")
    print(f"🚀 Training with LR = {lr}")
    print(f"{'='*60}")

    run_name = f"v0.6-mse-spearman-lr{lr:.0e}-15ep"
    output_dir = f"artifacts/models/cross-encoder-cv-jd-{run_name}"

    # Initialize model
    model = CrossEncoder(
        base_model,
        num_labels=1,
        default_activation_function=torch.nn.Sigmoid()
    )

    # Setup evaluator and data
    evaluator = CECorrelationEvaluator.from_input_examples(val_examples, name="val_spearman")
    train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)

    # Train
    model.fit(
        train_dataloader=train_dataloader,
        evaluator=evaluator,
        epochs=epochs,
        loss_fct=torch.nn.MSELoss(),
        optimizer_params={'lr': lr},
        warmup_steps=warmup_steps,
        output_path=output_dir,
        save_best_model=True,
        use_amp=True,
        max_grad_norm=1.0,
        show_progress_bar=True
    )

    # Load best model and compute metrics
    best_model = CrossEncoder(output_dir)
    val_metrics = compute_metrics(best_model, val_examples)
    test_metrics = compute_metrics(best_model, test_examples)

    print(f"\n✅ Completed LR={lr}")
    print(f"  Val LabelAcc:  {val_metrics['LabelAcc']:.4f} ({val_metrics['LabelAcc']*100:.2f}%)")
    print(f"  Test LabelAcc: {test_metrics['LabelAcc']:.4f} ({test_metrics['LabelAcc']*100:.2f}%)")

    # Store result
    result = {
        'learning_rate': float(lr),
        'run': run_name,
        'loss': 'MSE',
        'evaluator': 'Spearman',
        'epochs': epochs,
        'val': val_metrics,
        'test': test_metrics
    }
    all_results.append(result)

print(f"\n{'='*60}")
print(f"✅ Grid search completed!")
print(f"{'='*60}")

📊 Grid Search Configuration:
  Learning rates: [1e-05, 2e-05, 3e-05, 5e-05]
  Epochs: 15
  Batch size: 16
  Warmup steps: 877
  Total steps per run: 8775


🚀 Training with LR = 1e-05


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]


✅ Completed LR=5e-05
  Val LabelAcc:  0.6500 (65.00%)
  Test LabelAcc: 0.6515 (65.15%)

✅ Grid search completed!


## Results Summary

In [7]:
import pandas as pd

# Create summary table
summary_data = []
for result in all_results:
    summary_data.append({
        'Learning Rate': f"{result['learning_rate']:.0e}",
        'Val LabelAcc': f"{result['val']['LabelAcc']:.4f}",
        'Test LabelAcc': f"{result['test']['LabelAcc']:.4f}",
        'Val MAE': f"{result['val']['MAE']:.4f}",
        'Test MAE': f"{result['test']['MAE']:.4f}"
    })

df = pd.DataFrame(summary_data)
print("\n📊 Summary Table:")
print(df.to_string(index=False))

# Find best LR
best_result = max(all_results, key=lambda x: x['test']['LabelAcc'])
best_lr = best_result['learning_rate']
best_test_acc = best_result['test']['LabelAcc']

print(f"\n🏆 Best LR: {best_lr:.0e} with Test LabelAcc = {best_test_acc:.4f} ({best_test_acc*100:.2f}%)")
print(f"\n💡 Baseline (LR=2e-5): 61.60%")
print(f"   Improvement: {(best_test_acc - 0.6160)*100:+.2f}pp")


📊 Summary Table:
Learning Rate Val LabelAcc Test LabelAcc Val MAE Test MAE
        1e-05       0.6115        0.6105  9.6813  10.0865
        2e-05       0.5930        0.5825  9.9359  10.4430
        3e-05       0.6470        0.6380  9.1627   9.5916
        5e-05       0.6500        0.6515  9.3935   9.6212

🏆 Best LR: 5e-05 with Test LabelAcc = 0.6515 (65.15%)

💡 Baseline (LR=2e-5): 61.60%
   Improvement: +3.55pp


## Save Reports

In [8]:
import os
import json
from pathlib import Path

# Create reports directory
os.makedirs('artifacts/reports', exist_ok=True)

# Save individual reports
for result in all_results:
    lr = result['learning_rate']
    report = {
        'base_model': 'cross-encoder/ms-marco-MiniLM-L-12-v2',
        'dataset_version': 'v0.5',
        'dataset_size': {
            'train': len(train_examples),
            'val': len(val_examples),
            'test': len(test_examples),
            'total': len(train_examples) + len(val_examples) + len(test_examples)
        },
        'run': result['run'],
        'loss': result['loss'],
        'evaluator': result['evaluator'],
        'learning_rate': lr,
        'epochs': result['epochs'],
        'metrics': {
            'validation': {k: float(v) for k, v in result['val'].items()},
            'test': {k: float(v) for k, v in result['test'].items()}
        },
        'model_path': f'artifacts/models/cross-encoder-cv-jd-{result["run"]}'
    }

    report_path = f'artifacts/reports/fine_tune_cross_encoder_v0.6_mse_spearman_lr{lr:.0e}_15ep_report.json'
    with open(report_path, 'w') as f:
        json.dump(report, f, indent=2)
    print(f"✅ Report saved: {report_path}")

# Save summary report
summary_report = {
    'experiment': 'Phase 2.1: Learning Rate Grid Search',
    'base_model': 'cross-encoder/ms-marco-MiniLM-L-12-v2',
    'dataset_version': 'v0.5',
    'loss': 'MSELoss',
    'evaluator': 'Spearman',
    'epochs': 15,
    'batch_size': 16,
    'results': all_results,
    'best': {
        'learning_rate': float(best_lr),
        'test_label_acc': float(best_test_acc),
        'improvement_over_baseline': float((best_test_acc - 0.6160)*100)
    }
}

summary_path = 'artifacts/reports/fine_tune_cross_encoder_v0.6_lr_grid_search_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary_report, f, indent=2)
print(f"\n✅ Summary report saved: {summary_path}")

✅ Report saved: artifacts/reports/fine_tune_cross_encoder_v0.6_mse_spearman_lr1e-05_15ep_report.json
✅ Report saved: artifacts/reports/fine_tune_cross_encoder_v0.6_mse_spearman_lr2e-05_15ep_report.json
✅ Report saved: artifacts/reports/fine_tune_cross_encoder_v0.6_mse_spearman_lr3e-05_15ep_report.json
✅ Report saved: artifacts/reports/fine_tune_cross_encoder_v0.6_mse_spearman_lr5e-05_15ep_report.json

✅ Summary report saved: artifacts/reports/fine_tune_cross_encoder_v0.6_lr_grid_search_summary.json


## Save to Google Drive (Optional)

In [10]:
from google.colab import drive
import shutil
import os

drive.mount('/content/drive')

drive_base = "/content/drive/MyDrive/ai-recruiter"
os.makedirs(f"{drive_base}/models", exist_ok=True)
os.makedirs(f"{drive_base}/reports", exist_ok=True)

# Save all models
for result in all_results:
    src_dir = f'artifacts/models/cross-encoder-cv-jd-{result["run"]}'
    dest_dir = f"{drive_base}/models/cross-encoder-cv-jd-{result['run']}"
    if os.path.exists(dest_dir):
        shutil.rmtree(dest_dir)
    shutil.copytree(src_dir, dest_dir)
    print(f"✅ Saved model: {result['run']}")

# Copy all reports
for file in os.listdir('artifacts/reports'):
    if 'lr_grid_search' in file or 'v0.6_mse_spearman_lr' in file:
        src = f'artifacts/reports/{file}'
        dest = f"{drive_base}/reports/{file}"
        shutil.copy(src, dest)
        print(f"✅ Saved report: {file}")

print(f"\n✅ All models and reports saved to Google Drive!")

Mounted at /content/drive
✅ Saved model: v0.6-mse-spearman-lr1e-05-15ep
✅ Saved model: v0.6-mse-spearman-lr2e-05-15ep
✅ Saved model: v0.6-mse-spearman-lr3e-05-15ep
✅ Saved model: v0.6-mse-spearman-lr5e-05-15ep
✅ Saved report: fine_tune_cross_encoder_v0.6_lr_grid_search_summary.json
✅ Saved report: fine_tune_cross_encoder_v0.6_mse_spearman_lr5e-05_15ep_report.json
✅ Saved report: fine_tune_cross_encoder_v0.6_mse_spearman_lr2e-05_15ep_report.json
✅ Saved report: fine_tune_cross_encoder_v0.6_mse_spearman_lr3e-05_15ep_report.json
✅ Saved report: fine_tune_cross_encoder_v0.6_mse_spearman_lr1e-05_15ep_report.json

✅ All models and reports saved to Google Drive!
